In [0]:
import os

# Connecting and Mounting Blob Storage
storageAccountName = os.getenv("account_name")
storageAccountAccessKey =  os.getenv("account_key")
# sasToken = <sas-token>
blobContainerName = os.getenv("container_name")
mountPoint = "/mnt/data/"
if not any(mount.mountPoint == mountPoint for mount in dbutils.fs.mounts()):
  try:
    dbutils.fs.mount(
      source = "wasbs://{}@{}.blob.core.windows.net".format(blobContainerName, storageAccountName),
      mount_point = mountPoint,
      extra_configs = {'fs.azure.account.key.' + storageAccountName + '.blob.core.windows.net': storageAccountAccessKey}
    #   extra_configs = {'fs.azure.sas.' + blobContainerName + '.' + storageAccountName + '.blob.core.windows.net': sasToken}
    )
    print("mount succeeded!")
  except Exception as e:
    print("mount exception", e)



In [0]:
dbutils.fs.mounts()


[MountInfo(mountPoint='/databricks-datasets', source='databricks-datasets', encryptionType=''),
 MountInfo(mountPoint='/Volumes', source='UnityCatalogVolumes', encryptionType=''),
 MountInfo(mountPoint='/databricks/mlflow-tracking', source='databricks/mlflow-tracking', encryptionType=''),
 MountInfo(mountPoint='/databricks-results', source='databricks-results', encryptionType=''),
 MountInfo(mountPoint='/databricks/mlflow-registry', source='databricks/mlflow-registry', encryptionType=''),
 MountInfo(mountPoint='/Volume', source='DbfsReserved', encryptionType=''),
 MountInfo(mountPoint='/volumes', source='DbfsReserved', encryptionType=''),
 MountInfo(mountPoint='/mnt/data/', source='wasbs://glucosefeeds@glucosemonitoring.blob.core.windows.net', encryptionType=''),
 MountInfo(mountPoint='/', source='DatabricksRoot', encryptionType=''),
 MountInfo(mountPoint='/volume', source='DbfsReserved', encryptionType='')]

In [0]:
dbutils.fs.ls("/mnt/data/glucose_dimensional_model")

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-3543901295780034>, line 1
----> 1 dbutils.fs.ls("/mnt/data/glucose_dimensional_model")

File /databricks/python_shell/dbruntime/dbutils.py:364, in DBUtils.FSHandler.prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
    362 exc.__context__ = None
    363 exc.__cause__ = None
--> 364 raise exc

ExecutionError: An error occurred while calling o387.ls.
: java.io.FileNotFoundException: No such file or directory /mnt/data/glucose_dimensional_model
	at com.databricks.backend.daemon.data.client.DBFSV2.$anonfun$listStatus$2(DatabricksFileSystemV2.scala:193)
	at com.databricks.s3a.S3AExceptionUtils$.convertAWSExceptionToJavaIOException(DatabricksStreamUtils.scala:66)
	at com.databricks.backend.daemon.data.client.DBFSV2.$anonfun$listStatus$1(DatabricksFileSystemV2.scala:173)
	at com.databricks.logging.Usa

In [0]:
connectionString=os.getenv("EVENT_HUB_CONNECTION_STR")

FullConnectionString = f"{connectionString};EntityPath=increasing_trend_alert"

encrypted_conn_string = spark._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(FullConnectionString)

ehConfTrend = {
    "eventhubs.connectionString": encrypted_conn_string,
    "eventhubs.consumerGroup": "sparksql" 
}



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import datetime, timedelta
import json


# Initialize Spark session
spark = SparkSession.builder.appName("GlucoseTrendAnalysis").getOrCreate()

# # Loading the data from Azure Blob storage
# df_glucose = spark.read.parquet("dbfs:/mnt/data/glucose_dimensional_model/fact_glucose_reading.parquet")
# df_time = spark.read.parquet("dbfs:/mnt/data/glucose_dimensional_model/dim_time.parquet")

# # Registering the DataFrames as temporary views so they can be queried  with SQL
# df_glucose.createOrReplaceTempView("fact_glucose_reading")
# df_time.createOrReplaceTempView("dim_time")

# As we scheduled the job on the 1st of every month, we use datetime in other to ensure that we are getting the data for the previous month
current_date_for_sql = datetime.now()
first_day_of_current_month_for_sql = current_date_for_sql.replace(day=1)
last_day_of_previous_month_for_sql = first_day_of_current_month_for_sql - timedelta(days=1)
# Define the variables used in the f-string SQL query below
previous_month_int_sql = last_day_of_previous_month_for_sql.month
year_int_sql = last_day_of_previous_month_for_sql.year
print(f"Using Year={year_int_sql} and Month={previous_month_int_sql} in SQL query.")

print(f"Attempting to read data for Year: {year_to_read}, Month: {month_to_read_str}")
base_data_path = f"{mountPoint}/aggregates/"

try:
    daily_agg_df = spark.read.parquet(base_data_path) \
        .where(f"year = {year_to_read} AND month = '{month_to_read_str}'")

    print(f"Read {daily_agg_df.count()} records from {base_data_path} for {year_to_read}-{month_to_read_str}")
    if daily_agg_df.count() == 0:
         print("Warning: No data found for the previous month. Exiting.")
         dbutils.notebook.exit("No data found for previous month.")

    # --- ADD THIS LINE: Create Temporary View ---
    daily_agg_df.createOrReplaceTempView("daily_glucose_aggregates")
    print("Created temporary view 'daily_glucose_aggregates'")
    # --- END ADDED LINE ---

except Exception as e:
    print(f"Error reading Parquet data or creating view from {base_data_path}: {e}")
    raise

# # The SQL query to get the weekly average glucose levels for the previous month
# sql_query = f"""
# WITH WeeklyAverages AS (
#   SELECT 
#     user_id,
#     YEAR(full_date) AS year,
#     MONTH(full_date) AS month,
#     CEIL(DAY(full_date) / 7.0) AS week_of_month,
#     AVG(avg_glucose) AS avg_weekly_glucose
#   FROM fact_glucose_reading
#   JOIN dim_time ON fact_glucose_reading.date_key = dim_time.date_key
#   WHERE YEAR(full_date) = {year} AND MONTH(full_date) = {previous_month}
#   GROUP BY user_id, YEAR(full_date), MONTH(full_date), CEIL(DAY(full_date) / 7.0)
# ),
# ConsecutiveIncrease AS (
#   SELECT 
#     user_id,
#     year,
#     month,
#     week_of_month,
#     avg_weekly_glucose,
#     LAG(avg_weekly_glucose, 1) OVER (PARTITION BY user_id ORDER BY year, month, week_of_month) AS prev_week_glucose,
#     CASE 
#       WHEN avg_weekly_glucose > LAG(avg_weekly_glucose, 1) OVER (PARTITION BY user_id ORDER BY year, month, week_of_month) THEN 1
#       ELSE 0
#     END AS is_increasing
#   FROM WeeklyAverages
# ),
# UsersWithIncrease AS (
#   SELECT 
#     user_id
#   FROM ConsecutiveIncrease
#   WHERE month = {previous_month} AND year = {year}
#   GROUP BY user_id
#   HAVING SUM(is_increasing) = 4
# )
# SELECT 
#   a.user_id,
#   MAX(CASE WHEN a.week_of_month = 1 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week1,
#   MAX(CASE WHEN a.week_of_month = 2 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week2,
#   MAX(CASE WHEN a.week_of_month = 3 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week3,
#   MAX(CASE WHEN a.week_of_month = 4 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week4,
#   MAX(CASE WHEN a.week_of_month = 5 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week5
# FROM WeeklyAverages a
# JOIN UsersWithIncrease b ON a.user_id = b.user_id
# GROUP BY a.user_id
# """

# # Execute the SQL query
# result_df = spark.sql(sql_query)

# # Show the results
# result_df.show()


Using Year=2025 and Month=4 in SQL query.
Attempting to read data for Year: 2025, Month: 04
Read 50 records from /mnt/data//aggregates/ for 2025-04
Created temporary view 'daily_glucose_aggregates'


In [0]:
sql_query = f"""
WITH DailyDataWithDate AS (
    -- Reconstruct date from partition columns
    SELECT
        *,
        make_date(year, cast(month as int), cast(day as int)) as reading_date
    FROM daily_glucose_aggregates
    WHERE year = {year_int_sql} AND cast(month as int) = {previous_month_int_sql} -- Filter again for safety
),
WeeklyAverages AS (
  SELECT
    user_id,
    year,
    month, -- Keep original partition columns if needed
    CEIL(DAY(reading_date) / 7.0) AS week_of_month,
    AVG(avg_glucose) AS avg_weekly_glucose -- Use pre-aggregated daily avg
  FROM DailyDataWithDate
  GROUP BY user_id, year, month, CEIL(DAY(reading_date) / 7.0)
),
ConsecutiveIncrease AS (
  SELECT
    user_id,
    year,
    month,
    week_of_month,
    avg_weekly_glucose,
    LAG(avg_weekly_glucose, 1) OVER (PARTITION BY user_id ORDER BY year, cast(month as int), week_of_month) AS prev_week_glucose,
    CASE
      WHEN avg_weekly_glucose > LAG(avg_weekly_glucose, 1) OVER (PARTITION BY user_id ORDER BY year, cast(month as int), week_of_month) THEN 1
      ELSE 0
    END AS is_increasing
  FROM WeeklyAverages
),
UsersWithIncrease AS (
  SELECT
    user_id,
    year -- Include year for joining robustness
  FROM ConsecutiveIncrease
  WHERE cast(month as int) = {previous_month_int_sql} AND year = {year_int_sql}
  GROUP BY user_id, year
  -- Check if number of increases equals number of possible comparisons (weeks - 1)
  HAVING SUM(CASE WHEN is_increasing = 1 AND prev_week_glucose IS NOT NULL THEN 1 END) = (COUNT(*) - 1) AND COUNT(*) > 1
)
SELECT
  a.user_id,
  MAX(CASE WHEN a.week_of_month = 1 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week1,
  MAX(CASE WHEN a.week_of_month = 2 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week2,
  MAX(CASE WHEN a.week_of_month = 3 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week3,
  MAX(CASE WHEN a.week_of_month = 4 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week4,
  MAX(CASE WHEN a.week_of_month = 5 THEN a.avg_weekly_glucose ELSE NULL END) AS avg_week5
FROM WeeklyAverages a
JOIN UsersWithIncrease b ON a.user_id = b.user_id AND a.year = b.year
WHERE a.year = {year_int_sql} AND cast(a.month as int) = {previous_month_int_sql} 
GROUP BY a.user_id
ORDER BY a.user_id
"""
# --------------------------------------------------------------


# --------------------------------------------------------------
# Section 6: Execute Query and Show Results (Keep this)
# --------------------------------------------------------------
print("Executing SQL query...")
result_df = spark.sql(sql_query)
print("Query finished. Showing results:")
result_df.show(truncate=False)



Executing SQL query...
Query finished. Showing results:
+-------+---------+---------+---------+---------+---------+
|user_id|avg_week1|avg_week2|avg_week3|avg_week4|avg_week5|
+-------+---------+---------+---------+---------+---------+
+-------+---------+---------+---------+---------+---------+



In [0]:
# Checking if the DataFrame is non-empty
if result_df.count() > 0:
    increasing_trend_json_df = result_df.select(
        col("user_id").cast("string").alias("partitionKey"),
        to_json(struct(*[col(x) for x in result_df.columns])).alias("body")
    )

    # Preparing And Sending the data to Azure Event Hub
    increasing_trend_json_df\
        .write \
        .format("eventhubs") \
        .options(**ehConfTrend) \
        .save()
else:
    print(f"No users with 4 consecutive weeks of increasing glucose readings for the period {year}-{previous_month}")


In [0]:
dbutils.fs.unmount('/mnt/data')